# step to clean the dataset

## 1. Load and take stock

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("housing_dirty_500.csv")
print(df.dtypes)          # area/rooms/price will show as "object" — mixed text, not numeric
print(df.isnull().sum())
print(df.duplicated().sum())

## 2. Strip out the inconsistent text formatting

Each column has its own junk to remove: "sqft" and commas from area, currency symbols and commas from price, and spelled-out numbers from rooms.

In [ ]:
df["area"] = df["area"].astype(str).str.replace("sqft", "", regex=False)
df["area"] = df["area"].str.replace(",", "", regex=False).str.strip()
df["price"] = df["price"].astype(str).str.replace(r"[₦$,]", "", regex=True).str.strip()
  
word_to_num = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}
df["rooms"] = df["rooms"].astype(str).str.strip().replace(word_to_num)
df["rooms"] = df["rooms"].str.replace("rooms", "", regex=False).str.strip()

## 3. Convert everything to actual numbers

In [ ]:
for col in ["area", "rooms", "price"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

## 4. Fix impossible values

In [ ]:
df.loc[df["area"] <= 0, "area"] = np.nan
df.loc[df["price"] <= 0, "price"] = np.nan

## 5. Handle missing values

In [ ]:
for col in ["area", "rooms", "price"]:
    df[col] = df[col].fillna(df[col].median())

## 6. Handle outlier

In [ ]:
def cap_outliers_iqr(df, column):
    Q1, Q3 = df[column].quantile(0.25), df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower=lower, upper=upper)
    return df

for col in ["area", "price"]:
    df = cap_outliers_iqr(df, col)

## 7. Remove duplicates

In [ ]:
print("Before:", len(df))
df = df.drop_duplicates()
print("After:", len(df))

## 8. Final Chheck

In [ ]:
print(df.dtypes)          # should now all be numeric (float64)
print(df.isnull().sum())  # should all be 0
print(df.describe())      # eyeball the ranges for anything still off